In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK']='True'
import pandas as pd
import numpy as np
import spacy
import re
import time
import numpy as np
import time

import ast

In [ ]:
!python -m spacy download de_core_news_sm

2023-07-12 09:21:16.083553: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 112.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('de_core_news_sm')


In [ ]:
from spacy.matcher import PhraseMatcher
from spacy.matcher import Matcher
import re

In [ ]:
# Model data
model = {'en': spacy.load("en_core_web_sm"), 'de': spacy.load("de_core_news_sm")}

In [ ]:
#function to split string in the row by " | "
def split_form(row):
    row_split = row.split(" | ")
    return row_split

In [ ]:
# function to add ... for inspiration alternatives
def add_dots(row):
    list_out =[]
    if type(row)==str:
        row_list = row.split(" | ")
        for item in row_list:
            res = re.search(r"\...", item)
            if res is not None:
                list_out.append(item)
            else:
                list_out.append(item+"(...)")
        return " | ".join(list_out)

In [ ]:
#function to clean alternatives from pipe sign
def clean_str(row):
    array_out =[]
    if len(row)==0:
        return list()
    else:
        array_out =[i.strip(' |') for i in row]
        new_list = [x for x in array_out if x != '']
        return new_list

In [ ]:
#function to split words and sentences into two different dataframes for pipe/no-pipe in word_type
def split_words_sentences_pipe(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []

    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []

    list_false_positives_word = []
    list_false_positives_sentence = []


    for index, row in df.iterrows():
        word = row["Lemma"]
        word_list = word.split() # Split `a_string` by whitespace.
        number_of_words = len(word_list)
        word_type = row["Word_Type"]
        res = re.search(r"|", word_type)

        if number_of_words == 1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])
            list_false_positives_word.append(row["False_Positives"])


        else:
          if res is not None:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])
            list_false_positives_word.append(row["False_Positives"])

          else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])
            list_false_positives_sentence.append(row["False_Positives"])

    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word, list_false_positives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split", "False_Positives"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_sg_all_sentence, list_pl_sentence, list_pl_all_sentence, list_alternatives_sentence, list_false_positives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split", "False_Positives"])

    return df_word, df_sentence


In [ ]:
#function to separate one term binary pronouns excluding / (for example, exclude he/she)
def separate_binary_pronouns(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []

    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []


    for index, row in df.iterrows():
        word = row["Lemma"]
        span = re.search("/", word)

        if len(word.rsplit(" "))==1 and len(word.rsplit("/"))==1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])

    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])

    return df_word




In [ ]:
#function to split words and sentences into two different dataframes for singular/plural dataframes. Used for gendered category to handle multiterms pronouns
def split_binary_pronouns(df):
    list_word = []
    list_sentence = []

    list_word_type_word = []
    list_word_type_sentence = []

    list_category_word = []
    list_category_sentence = []

    list_primary_subcategory_word = []
    list_primary_subcategory_sentence = []

    list_secondary_subcategory_word = []
    list_secondary_subcategory_sentence = []

    list_sg_all_word = []
    list_sg_all_sentence = []

    list_pl_word = []
    list_pl_sentence = []

    list_pl_all_word = []
    list_pl_all_sentence = []

    list_alternatives_word = []
    list_alternatives_sentence = []


    for index, row in df.iterrows():
        word = row["Lemma"]
        span = re.search("/", word)


        if len(word.rsplit(" "))==1 and len(word.rsplit("/"))==1:
            list_word.append(word)
            list_word_type_word.append(row["Word_Type"])
            list_category_word.append(row["Category"])
            list_primary_subcategory_word.append(row["Primary_subcategory"])
            list_secondary_subcategory_word.append(row["Secondary_subcategory"])
            list_sg_all_word.append(row["Sg_all_split"])
            list_pl_word.append(row["Pl_all"])
            list_pl_all_word.append(row["Pl_all_split"])
            list_alternatives_word.append(row["Alt_split"])

        elif type(span) == re.Match and len(word.rsplit("/"))==2:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_secondary_subcategory_sentence.append(row["Secondary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])

        else:
            list_sentence.append(word)
            list_word_type_sentence.append(row["Word_Type"])
            list_category_sentence.append(row["Category"])
            list_primary_subcategory_sentence.append(row["Primary_subcategory"])
            list_secondary_subcategory_sentence.append(row["Secondary_subcategory"])
            list_sg_all_sentence.append(row["Sg_all_split"])
            list_pl_sentence.append(row["Pl_all"])
            list_pl_all_sentence.append(row["Pl_all_split"])
            list_alternatives_sentence.append(row["Alt_split"])

    df_word = pd.DataFrame(list(zip(list_word, list_word_type_word, list_category_word, list_primary_subcategory_word, list_secondary_subcategory_word, list_sg_all_word, list_pl_word, list_pl_all_word, list_alternatives_word)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory","Secondary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])
    df_sentence = pd.DataFrame(list(zip(list_sentence, list_word_type_sentence, list_category_sentence, list_primary_subcategory_sentence, list_secondary_subcategory_sentence, list_sg_all_sentence, list_pl_sentence, list_pl_all_sentence, list_alternatives_sentence)), columns=["Lemma", "Word_Type", "Category", "Primary_subcategory", "Secondary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split"])

    return df_word, df_sentence




In [ ]:
#function to covert alternatives into right format
def replace_tolist(file_csv):

    # creating a variable and storing the search text
    search_text = "['']"

    # creating a variable and storing the replace text
    replace_text = "[]"

    # Opening our text file in read only
    # mode using the open() function
    with open(file_csv, 'r') as file:

        # Reading the content of the file
        # using the read() function and storing
        # them in a new variable
        data = file.read()

        # Searching and replacing the text
        # using the replace() function
        data = data.replace(search_text, replace_text)

    with open(file_csv, 'w') as file:
        # Writing the replaced data in our
        # text file
        file.write(data)
    return file_csv



In [ ]:
# use if you run scripts on google colabs
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


https://drive.google.com/file/d/192EXnDim3b5KBbFr67zvXKapNM8xz0xs/view?usp=share_link

In [ ]:
# read csv file
df_rules = pd.read_csv("/content/drive/MyDrive/rules_11JUL_German.csv")

In [ ]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2636 entries, 0 to 2635
Data columns (total 26 columns):
 #   Column                                                                                  Non-Null Count  Dtype 
---  ------                                                                                  --------------  ----- 
 0   Lemma                                                                                   2635 non-null   object
 1   Word_Type                                                                               2636 non-null   object
 2   Priority                                                                                17 non-null     object
 3   Category                                                                                2636 non-null   object
 4   Primary_subcategory                                                                     2636 non-null   object
 5   Secondary_subcategory                                                       

In [ ]:
#drop row if all value in row are missing
df_rules.dropna(how='all', inplace=True)

In [ ]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2636 entries, 0 to 2635
Data columns (total 26 columns):
 #   Column                                                                                  Non-Null Count  Dtype 
---  ------                                                                                  --------------  ----- 
 0   Lemma                                                                                   2635 non-null   object
 1   Word_Type                                                                               2636 non-null   object
 2   Priority                                                                                17 non-null     object
 3   Category                                                                                2636 non-null   object
 4   Primary_subcategory                                                                     2636 non-null   object
 5   Secondary_subcategory                                                       

In [ ]:
df_rules["Primary_subcategory"].unique()

array(['ableism', 'advanced_abbreviation', 'advanced_ability',
       'advanced_ableism_base', 'advanced_anglicism',
       'advanced_binary_pronouns', 'advanced_formality',
       'advanced_hidden_image', 'advanced_hollow',
       'advanced_homophobia_base', 'advanced_offensive_language_base',
       'advanced_racism_base', 'advanced_sexism_base',
       'advanced_titles_base', 'age', 'age_old', 'age_young', 'agentic',
       'antimuslim', 'antisemitism', 'belief', 'classism',
       'cognitive_ability', 'cognitive_perception', 'color', 'communal',
       'culture', 'd_and_i', 'exaggerating', 'female_stereotype',
       'filler', 'function_base', 'function', 'gender_identity',
       'hearing', 'homophobia', 'leadership', 'male_stereotype',
       'medical_state', 'mental_wellbeing', 'migration',
       'military_source', 'mobility', 'nazi_language',
       'offensive_language', 'racism', 'racist_source', 'sexism',
       'sexual_orientation', 'speech', 'sports_terms', 'titles',
     

### Drop duplicates

In [ ]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Identity_first,...,Alt_Pl_people_first,3rd_party_alternatives,False_Positives,Sample_Sentences,Rule copy,3rd_party_source,ToClarify,Notes,Write a story about a young girl who discovers a hidden magical world in her backyard.,Prompt
621,Lesbe,s,NaN,openly_discriminating,advanced_homophobia_base,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1391,Lesbe,s,NaN,inclusive,d_and_i,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"je nachdem wer den Begriff verwendet, ist Acht...",NaN,NaN


In [ ]:
#get index of duplicates
i1 = df_rules[((df_rules.Lemma == 'Lesbe'))].index

In [ ]:
#get index of duplicates
i2 = df_rules[((df_rules.Lemma == 'LGBTQIA+'))].index

In [ ]:
#get index of duplicates
i3 = df_rules[((df_rules.Lemma == 'Zwischengeschlechtlichkeit'))].index

In [ ]:
#drop row with one duplicate
df_rules = df_rules.drop(i1[1])

In [ ]:
#drop row with one duplicate
#df_rules = df_rules.drop(i2[0])

In [ ]:
#drop row with one duplicate
#df_rules = df_rules.drop(i3[0])

In [ ]:
df_rules["Lemma"].duplicated().sum()

1

In [ ]:
df_rules[((df_rules.Lemma == 'Lesbe'))]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Identity_first,...,Alt_Pl_people_first,3rd_party_alternatives,False_Positives,Sample_Sentences,Rule copy,3rd_party_source,ToClarify,Notes,Write a story about a young girl who discovers a hidden magical world in her backyard.,Prompt
621,Lesbe,s,NaN,openly_discriminating,advanced_homophobia_base,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2636 entries, 0 to 2635
Data columns (total 26 columns):
 #   Column                                                                                  Non-Null Count  Dtype 
---  ------                                                                                  --------------  ----- 
 0   Lemma                                                                                   2635 non-null   object
 1   Word_Type                                                                               2636 non-null   object
 2   Priority                                                                                17 non-null     object
 3   Category                                                                                2636 non-null   object
 4   Primary_subcategory                                                                     2636 non-null   object
 5   Secondary_subcategory                                                       

In [ ]:
#find NAN to convert into the right format
df_rules[df_rules["Lemma"].isna()]

In [ ]:
#convert NaN to string "N/A" in lemmas
df_rules.Lemma.fillna("N/A", inplace=True)

In [ ]:
#check if it was converted
df_rules[df_rules["Lemma"].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Identity_first,...,Alt_Pl_people_first,3rd_party_alternatives,False_Positives,Sample_Sentences,Rule copy,3rd_party_source,ToClarify,Notes,Write a story about a young girl who discovers a hidden magical world in her backyard.,Prompt


### Check if subcategory name is missing and set the parent category name to subcategory if the subcategory name is missing

In [ ]:
#check missing values in Primary subcategory
df_rules[df_rules['Primary_subcategory'].isna()]

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Identity_first,...,Alt_Pl_people_first,3rd_party_alternatives,False_Positives,Sample_Sentences,Rule copy,3rd_party_source,ToClarify,Notes,Write a story about a young girl who discovers a hidden magical world in her backyard.,Prompt


In [ ]:
#fill the missing values in "primary subcategory" by values from "Category"
df_rules.Primary_subcategory.fillna(df_rules.Category, inplace=True)

In [ ]:
#check if the values are filled
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2636 entries, 0 to 2635
Data columns (total 26 columns):
 #   Column                                                                                  Non-Null Count  Dtype 
---  ------                                                                                  --------------  ----- 
 0   Lemma                                                                                   2636 non-null   object
 1   Word_Type                                                                               2636 non-null   object
 2   Priority                                                                                17 non-null     object
 3   Category                                                                                2636 non-null   object
 4   Primary_subcategory                                                                     2636 non-null   object
 5   Secondary_subcategory                                                       

### add (...) to
Refraimed columns:
Alt_Pl_people_first
Alt_Pl_reframed

Alt_Sg_people_first
Alt_Sg_reframed

In [ ]:
# add (...) for inspiration alternatives
#df_rules["Alt_Pl_reframed"]=df_rules["Alt_Pl_reframed"].map(add_dots)

In [ ]:
# add (...) for inspiration alternatives
df_rules["Alt_Pl_people_first"]=df_rules["Alt_Pl_people_first"].map(add_dots)

In [ ]:
# add (...) for inspiration alternatives
df_rules["Alt_Sg_people_first"]=df_rules["Alt_Sg_people_first"].map(add_dots)

In [ ]:
# add (...) for inspiration alternatives
df_rules["Alt_Sg_reframed"]=df_rules["Alt_Sg_reframed"].map(add_dots)

## Alternatives Single

### Single click+rephrase. Important order of columns with alternatives

In [ ]:
#order for plural form of alternatives provided by Tracey
df_rules["Sg_all"]= df_rules.loc[:, ("Identity_first", "Alt_Sg_Replacement", "Alt_Field", "Alt_Sg_/_and_inclusive_form", "Alt_Pl_collective_noun", "Alt_Sg_people_first", "Alt_Sg_reframed")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [ ]:
# split alternatives in the row by " | " applying function split_form to convert string alternatives into array
df_rules["Sg_all_split"]=df_rules["Sg_all"].map(split_form)

## Alternatives Plural

### Single click + rephrase. Important order of columns with alternatives

In [ ]:
#order for plural form of alternatives provided by Tracey
df_rules["Pl_all"]= df_rules.loc[:, ("Identity_first_pl", "Alt_Pl", "Alt_Pl_pair_and_inclusive_form", "Alt_Pl_collective_noun", "Alt_Field", "Alt_Pl_people_first")].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)

In [ ]:
# split alternatives in the row by " | " applying function split_form
df_rules["Pl_all_split"]=df_rules["Pl_all"].map(split_form)

In [ ]:
df_rules.tail(20)

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Identity_first,...,Rule copy,3rd_party_source,ToClarify,Notes,Write a story about a young girl who discovers a hidden magical world in her backyard.,Prompt,Sg_all,Sg_all_split,Pl_all,Pl_all_split
2616,Fischkopf,s,NaN,openly_discriminating,xenophobia,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-,[-],-,[-]
2617,Fischkopp,s,NaN,openly_discriminating,xenophobia,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-,[-],-,[-]
2618,Inselaffe,s,NaN,openly_discriminating,xenophobia,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-,[-],-,[-]
2619,Itacker,s,NaN,openly_discriminating,xenophobia,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-,[-],-,[-]
2620,Iwan,s,NaN,openly_discriminating,xenophobia,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-,[-],-,[-]
2621,Jugo,s,NaN,openly_discriminating,xenophobia,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-,[-],-,[-]
2622,Kanacker,s,NaN,openly_discriminating,xenophobia,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-,[-],-,[-]
2623,Piefke,s,NaN,openly_discriminating,xenophobia,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-,[-],-,[-]
2624,Polack,s,NaN,openly_discriminating,xenophobia,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-,[-],-,[-]
2625,Polacke,s,NaN,openly_discriminating,xenophobia,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-,[-],-,[-]


### Merge alternatives into one column

In [ ]:
source_col_loc = df_rules.columns.get_loc('Alt_Field') # column position starts from 0

In [ ]:
#merge alternatives in one column by sign  " | "
df_rules['alternatives'] = df_rules.iloc[:,source_col_loc:source_col_loc+11].apply(
    lambda x: " | ".join(x.dropna().astype(str)), axis=1)


In [ ]:
# split alternatives in the row by " | " applying function split_form
df_rules["Alt_split"]=df_rules["alternatives"].map(split_form)

In [ ]:
df_rules["False_Positives"] = df_rules["False_Positives"].astype(str).replace("nan", "")

In [ ]:
# split false positives in the row by " | " applying function split_form
df_rules["False_Positives"]=df_rules["False_Positives"].map(split_form)

In [ ]:
df_rules.head(10)

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Identity_first,...,ToClarify,Notes,Write a story about a young girl who discovers a hidden magical world in her backyard.,Prompt,Sg_all,Sg_all_split,Pl_all,Pl_all_split,alternatives,Alt_split
0,Spast,s,NaN,openly_discriminating,ableism,NaN,-,Spastiker,NaN,NaN,...,NaN,Solche Ausdrücke lösen negative Assoziationen ...,NaN,NaN,"Spastiker | - | Person, die mit Spasmen lebt(....","[Spastiker, -, Person, die mit Spasmen lebt(.....",-,[-],"- | Spastiker | Person, die mit Spasmen lebt(....","[-, Spastiker, Person, die mit Spasmen lebt(....."
1,Spasti,s,NaN,openly_discriminating,ableism,NaN,-,Spastiker,NaN,NaN,...,NaN,NaN,NaN,NaN,"Spastiker | - | Person, die mit Spasmen lebt(....","[Spastiker, -, Person, die mit Spasmen lebt(.....",-,[-],"- | Spastiker | Person, die mit Spasmen lebt(....","[-, Spastiker, Person, die mit Spasmen lebt(....."
2,Albino,s,NaN,openly_discriminating,ableism,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-,[-],-,[-],-,[-]
3,Hungerhaken,s,NaN,openly_discriminating,ableism,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-,[-],-,[-],-,[-]
4,Krisperl,s,NaN,openly_discriminating,ableism,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-,[-],-,[-],-,[-]
5,Lallkopf,s,NaN,openly_discriminating,ableism,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-,[-],-,[-],-,[-]
6,Schielauge,s,NaN,openly_discriminating,ableism,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,- | Person mit Gleichgewichtsstörung des Augen...,"[-, Person mit Gleichgewichtsstörung des Augen...",-,[-],- | Person mit Gleichgewichtsstörung des Augen...,"[-, Person mit Gleichgewichtsstörung des Augen..."
7,Schwachmat,s,NaN,openly_discriminating,ableism,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-,[-],-,[-],-,[-]
8,Schwachsinniger,s,NaN,openly_discriminating,ableism,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-,[-],-,[-],-,[-]
9,Spagmat,s,NaN,openly_discriminating,ableism,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-,[-],-,[-],-,[-]


In [ ]:
df_rules.head()

,Lemma,Word_Type,Priority,Category,Primary_subcategory,Secondary_subcategory,Alt_Field,Alt_Sg_Replacement,Alt_Sg_/_and_inclusive_form,Identity_first,...,ToClarify,Notes,Write a story about a young girl who discovers a hidden magical world in her backyard.,Prompt,Sg_all,Sg_all_split,Pl_all,Pl_all_split,alternatives,Alt_split
0,Spast,s,NaN,openly_discriminating,ableism,NaN,-,Spastiker,NaN,NaN,...,NaN,Solche Ausdrücke lösen negative Assoziationen ...,NaN,NaN,"Spastiker | - | Person, die mit Spasmen lebt(....","[Spastiker, -, Person, die mit Spasmen lebt(.....",-,[-],"- | Spastiker | Person, die mit Spasmen lebt(....","[-, Spastiker, Person, die mit Spasmen lebt(....."
1,Spasti,s,NaN,openly_discriminating,ableism,NaN,-,Spastiker,NaN,NaN,...,NaN,NaN,NaN,NaN,"Spastiker | - | Person, die mit Spasmen lebt(....","[Spastiker, -, Person, die mit Spasmen lebt(.....",-,[-],"- | Spastiker | Person, die mit Spasmen lebt(....","[-, Spastiker, Person, die mit Spasmen lebt(....."
2,Albino,s,NaN,openly_discriminating,ableism,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-,[-],-,[-],-,[-]
3,Hungerhaken,s,NaN,openly_discriminating,ableism,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-,[-],-,[-],-,[-]
4,Krisperl,s,NaN,openly_discriminating,ableism,NaN,-,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,-,[-],-,[-],-,[-]


### cleaning alternatives

In [ ]:
# strip alternatives
df_rules["Alt_split"]=df_rules["Alt_split"].map(clean_str)

In [ ]:
# strip alternatives
df_rules["Sg_all_split"]=df_rules["Sg_all_split"].map(clean_str)

In [ ]:
# strip alternatives
df_rules["Pl_all_split"]=df_rules["Pl_all_split"].map(clean_str)

In [ ]:
# strip false positives
df_rules["False_Positives"]=df_rules["False_Positives"].map(clean_str)

### Create subsets of dataframes

In [ ]:
df_rules = df_rules[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all", "Pl_all_split", "Alt_split", "False_Positives"]]

### Check duplicates&homonyms

In [ ]:
df_rules[df_rules["Lemma"].duplicated(keep = False)]

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
621,Lesbe,s,openly_discriminating,advanced_homophobia_base,[-],-,[-],[-],[]
1391,Lesbe,s,inclusive,d_and_i,[],,[],[],[]


## Binary_pronouns

In [ ]:
df_binary_pronouns = df_rules[df_rules["Primary_subcategory"]=="advanced_binary_pronouns"]

In [ ]:
df_binary_pronouns.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16 entries, 511 to 526
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                16 non-null     object
 1   Word_Type            16 non-null     object
 2   Category             16 non-null     object
 3   Primary_subcategory  16 non-null     object
 4   Sg_all_split         16 non-null     object
 5   Pl_all               16 non-null     object
 6   Pl_all_split         16 non-null     object
 7   Alt_split            16 non-null     object
 8   False_Positives      16 non-null     object
dtypes: object(9)
memory usage: 1.2+ KB


In [ ]:
df_binary_pronouns

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
511,er,s,gendered,advanced_binary_pronouns,"[du, Sie, ((Vorname)), ((Vorname Nachname)), s...",,[],"[du, Sie, ((Vorname)), ((Vorname Nachname)), s...",[]
512,ihm,~s,gendered,advanced_binary_pronouns,"[dir, Ihnen, ((Vorname)), ((Vorname Nachname))...",,[],"[dir, Ihnen, ((Vorname)), ((Vorname Nachname))...",[]
513,ihn,~s,gendered,advanced_binary_pronouns,"[dich, Sie, ((Vorname)), ((Vorname Nachname)),...",,[],"[dich, Sie, ((Vorname)), ((Vorname Nachname)),...",[]
514,ihr,~s,gendered,advanced_binary_pronouns,"[dir, dein, Ihnen, Ihr, Ihr, ((Vorname)), ((Vo...",,[],"[dir, dein, Ihnen, Ihr, Ihr, ((Vorname)), ((Vo...",[]
515,ihre,~s,gendered,advanced_binary_pronouns,"[deine, Ihre, ((Vorname))s, ((Vorname Nachname...",,[],"[deine, Ihre, ((Vorname))s, ((Vorname Nachname...",[]
516,ihrem,~s,gendered,advanced_binary_pronouns,"[deinem, Ihrem, ((Vorname))s, ((Vorname Nachna...",,[],"[deinem, Ihrem, ((Vorname))s, ((Vorname Nachna...",[]
517,ihren,~s,gendered,advanced_binary_pronouns,"[deinen, Ihren, ((Vorname))s, ((Vorname Nachna...",,[],"[deinen, Ihren, ((Vorname))s, ((Vorname Nachna...",[]
518,ihrer,~s,gendered,advanced_binary_pronouns,"[deiner, Ihrer, ((Vorname))s, ((Vorname Nachna...",,[],"[deiner, Ihrer, ((Vorname))s, ((Vorname Nachna...",[]
519,ihres,~s,gendered,advanced_binary_pronouns,"[deines, Ihres, ((Vorname))s, ((Vorname Nachna...",,[],"[deines, Ihres, ((Vorname))s, ((Vorname Nachna...",[]
520,sein,~s,gendered,advanced_binary_pronouns,"[dein, Ihr, ((Vorname))s, ((Vorname Nachname))...",,[],"[dein, Ihr, ((Vorname))s, ((Vorname Nachname))...",[]


In [ ]:
# separate one term binary pronouns
#df_binary_pronouns_word =separate_binary_pronouns(df_binary_pronouns)

In [ ]:
#check the results
#df_binary_pronouns_word

In [ ]:
#create subset of dataframe to drop unnecessary columns
df_binary_pronouns = df_binary_pronouns[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Sg_all_split", "Pl_all_split", "Alt_split"]]

In [ ]:
#save one term binary_pronouns to csv file after preprocessing
df_binary_pronouns.to_csv("gendered_singular_they.csv", index=False)

In [ ]:
df_binary_pronouns.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 16 entries, 511 to 526
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                16 non-null     object
 1   Word_Type            16 non-null     object
 2   Category             16 non-null     object
 3   Primary_subcategory  16 non-null     object
 4   Sg_all_split         16 non-null     object
 5   Pl_all_split         16 non-null     object
 6   Alt_split            16 non-null     object
dtypes: object(7)
memory usage: 1.0+ KB


In [ ]:
df_binary_pronouns

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all_split,Alt_split
511,er,s,gendered,advanced_binary_pronouns,"[du, Sie, ((Vorname)), ((Vorname Nachname)), s...",[],"[du, Sie, ((Vorname)), ((Vorname Nachname)), s..."
512,ihm,~s,gendered,advanced_binary_pronouns,"[dir, Ihnen, ((Vorname)), ((Vorname Nachname))...",[],"[dir, Ihnen, ((Vorname)), ((Vorname Nachname))..."
513,ihn,~s,gendered,advanced_binary_pronouns,"[dich, Sie, ((Vorname)), ((Vorname Nachname)),...",[],"[dich, Sie, ((Vorname)), ((Vorname Nachname)),..."
514,ihr,~s,gendered,advanced_binary_pronouns,"[dir, dein, Ihnen, Ihr, Ihr, ((Vorname)), ((Vo...",[],"[dir, dein, Ihnen, Ihr, Ihr, ((Vorname)), ((Vo..."
515,ihre,~s,gendered,advanced_binary_pronouns,"[deine, Ihre, ((Vorname))s, ((Vorname Nachname...",[],"[deine, Ihre, ((Vorname))s, ((Vorname Nachname..."
516,ihrem,~s,gendered,advanced_binary_pronouns,"[deinem, Ihrem, ((Vorname))s, ((Vorname Nachna...",[],"[deinem, Ihrem, ((Vorname))s, ((Vorname Nachna..."
517,ihren,~s,gendered,advanced_binary_pronouns,"[deinen, Ihren, ((Vorname))s, ((Vorname Nachna...",[],"[deinen, Ihren, ((Vorname))s, ((Vorname Nachna..."
518,ihrer,~s,gendered,advanced_binary_pronouns,"[deiner, Ihrer, ((Vorname))s, ((Vorname Nachna...",[],"[deiner, Ihrer, ((Vorname))s, ((Vorname Nachna..."
519,ihres,~s,gendered,advanced_binary_pronouns,"[deines, Ihres, ((Vorname))s, ((Vorname Nachna...",[],"[deines, Ihres, ((Vorname))s, ((Vorname Nachna..."
520,sein,~s,gendered,advanced_binary_pronouns,"[dein, Ihr, ((Vorname))s, ((Vorname Nachname))...",[],"[dein, Ihr, ((Vorname))s, ((Vorname Nachname))..."


In [ ]:
replace_tolist(r"gendered_singular_they.csv")

'gendered_singular_they.csv'

In [ ]:
#test read of dataframe
df_test = pd.read_csv("gendered_singular_they.csv")

In [ ]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                16 non-null     object
 1   Word_Type            16 non-null     object
 2   Category             16 non-null     object
 3   Primary_subcategory  16 non-null     object
 4   Sg_all_split         16 non-null     object
 5   Pl_all_split         16 non-null     object
 6   Alt_split            16 non-null     object
dtypes: object(7)
memory usage: 1.0+ KB


In [ ]:
df_test

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all_split,Alt_split
0,er,s,gendered,advanced_binary_pronouns,"['du', 'Sie', '((Vorname))', '((Vorname Nachna...",[],"['du', 'Sie', '((Vorname))', '((Vorname Nachna..."
1,ihm,~s,gendered,advanced_binary_pronouns,"['dir', 'Ihnen', '((Vorname))', '((Vorname Nac...",[],"['dir', 'Ihnen', '((Vorname))', '((Vorname Nac..."
2,ihn,~s,gendered,advanced_binary_pronouns,"['dich', 'Sie', '((Vorname))', '((Vorname Nach...",[],"['dich', 'Sie', '((Vorname))', '((Vorname Nach..."
3,ihr,~s,gendered,advanced_binary_pronouns,"['dir', 'dein', 'Ihnen', 'Ihr', 'Ihr', '((Vorn...",[],"['dir', 'dein', 'Ihnen', 'Ihr', 'Ihr', '((Vorn..."
4,ihre,~s,gendered,advanced_binary_pronouns,"['deine', 'Ihre', '((Vorname))s', '((Vorname N...",[],"['deine', 'Ihre', '((Vorname))s', '((Vorname N..."
5,ihrem,~s,gendered,advanced_binary_pronouns,"['deinem', 'Ihrem', '((Vorname))s', '((Vorname...",[],"['deinem', 'Ihrem', '((Vorname))s', '((Vorname..."
6,ihren,~s,gendered,advanced_binary_pronouns,"['deinen', 'Ihren', '((Vorname))s', '((Vorname...",[],"['deinen', 'Ihren', '((Vorname))s', '((Vorname..."
7,ihrer,~s,gendered,advanced_binary_pronouns,"['deiner', 'Ihrer', '((Vorname))s', '((Vorname...",[],"['deiner', 'Ihrer', '((Vorname))s', '((Vorname..."
8,ihres,~s,gendered,advanced_binary_pronouns,"['deines', 'Ihres', '((Vorname))s', '((Vorname...",[],"['deines', 'Ihres', '((Vorname))s', '((Vorname..."
9,sein,~s,gendered,advanced_binary_pronouns,"['dein', 'Ihr', '((Vorname))s', '((Vorname Nac...",[],"['dein', 'Ihr', '((Vorname))s', '((Vorname Nac..."


### Remove one term binary_pronouns from dataframe for the next steps pre-processing

In [ ]:
#remove binary pronouns
binary_list = list(df_binary_pronouns["Lemma"])

In [ ]:
len(binary_list)

16

In [ ]:
df_rules = df_rules.query("Lemma != @binary_list")

In [ ]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2620 entries, 0 to 2635
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                2620 non-null   object
 1   Word_Type            2620 non-null   object
 2   Category             2620 non-null   object
 3   Primary_subcategory  2620 non-null   object
 4   Sg_all_split         2620 non-null   object
 5   Pl_all               2620 non-null   object
 6   Pl_all_split         2620 non-null   object
 7   Alt_split            2620 non-null   object
 8   False_Positives      2620 non-null   object
dtypes: object(9)
memory usage: 204.7+ KB


In [ ]:
df_rules["Primary_subcategory"].unique()

array(['ableism', 'advanced_abbreviation', 'advanced_ability',
       'advanced_ableism_base', 'advanced_anglicism',
       'advanced_formality', 'advanced_hidden_image', 'advanced_hollow',
       'advanced_homophobia_base', 'advanced_offensive_language_base',
       'advanced_racism_base', 'advanced_sexism_base',
       'advanced_titles_base', 'age', 'age_old', 'age_young', 'agentic',
       'antimuslim', 'antisemitism', 'belief', 'classism',
       'cognitive_ability', 'cognitive_perception', 'color', 'communal',
       'culture', 'd_and_i', 'exaggerating', 'female_stereotype',
       'filler', 'function_base', 'function', 'gender_identity',
       'hearing', 'homophobia', 'leadership', 'male_stereotype',
       'medical_state', 'mental_wellbeing', 'migration',
       'military_source', 'mobility', 'nazi_language',
       'offensive_language', 'racism', 'racist_source', 'sexism',
       'sexual_orientation', 'speech', 'sports_terms', 'titles',
       'titles_base', 'transphobia', 'vi

## Acronyms (abbreviations)

In [ ]:
df_rules["Word_Type"].unique()

array(['s', 'acr', 'abbr', '-acr', 'v', '|a|s|v', '||=a', 'a', 'a|v',
       '=a|s', 's|v|a', 's|v', 'v|s', 'a|s', 's||s|v', 'v||s||s', 'v|a|s',
       'a|s|v', 'v|a', 'a|=a', 'v|', 'v||s', '|s|v', '|a|s', '|s', '|=s',
       '||=s', '||s', '||s|v', 's|s', '||a|s', 'v||s|s', 'a|a', '|a',
       'adv|a', '|a|s|', '|s|', 'adv||s', '~|~|~|~', '~|~|~|~|~s',
       'adv|v', '|s||v', '|s||v|v|s|', 'v|||a|s', 'v||a|s', '~v||~v',
       's|||s', 'adv', 's|v|s', 'a||a', '=s', '|v|s||a|s', 'v|s||a|s',
       's||a|s|v', 'v|s||s', '||s|||s', 'IIs', 'adv|adv', 'adv|a|v|adv',
       's||s', '|a|s|=a', '~s|=', 'v|adv||s', 'adv||s|v', 's|v||s',
       '~s|s', 'v|a|s|', '|s||s|v', '||s|a', 's|a|s', '||a|s|v||a|s',
       'adv|a||a', 'a||s', 'v|||s', '|a|s|v||||s'], dtype=object)

In [ ]:
df_rules[df_rules["Word_Type"]=="IIs"]

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
1536,über dem Durchschnitt,IIs,style,exaggerating,"[viel, ausgesprochen, hochgradig, mehr als, zu...",,[],"[viel, ausgesprochen, hochgradig, mehr als, zu...",[]


In [ ]:
# create a dataframe with abbreviations
df_abbr = df_rules[(df_rules["Word_Type"]=='abbr') | (df_rules["Word_Type"]=='acr')| (df_rules["Word_Type"]=='-acr')]

In [ ]:
df_abbr.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 301 entries, 15 to 1740
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                301 non-null    object
 1   Word_Type            301 non-null    object
 2   Category             301 non-null    object
 3   Primary_subcategory  301 non-null    object
 4   Sg_all_split         301 non-null    object
 5   Pl_all               301 non-null    object
 6   Pl_all_split         301 non-null    object
 7   Alt_split            301 non-null    object
 8   False_Positives      301 non-null    object
dtypes: object(9)
memory usage: 23.5+ KB


In [ ]:
#create subset of dataframe to drop unnecessary columns
df_abbr = df_abbr[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [ ]:
#save acronyms (abbreviations) to csv file after preprocessing
df_abbr.to_csv("abbreviations.csv", index=False)

### Remove acronymes (abbr) from dataframe for the next steps pre-processing

In [ ]:
#remove abbr
abbr_list = list(df_abbr["Lemma"])

In [ ]:
df_rules = df_rules.query("Lemma != @abbr_list")

In [ ]:
df_rules.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 2319 entries, 0 to 2635
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                2319 non-null   object
 1   Word_Type            2319 non-null   object
 2   Category             2319 non-null   object
 3   Primary_subcategory  2319 non-null   object
 4   Sg_all_split         2319 non-null   object
 5   Pl_all               2319 non-null   object
 6   Pl_all_split         2319 non-null   object
 7   Alt_split            2319 non-null   object
 8   False_Positives      2319 non-null   object
dtypes: object(9)
memory usage: 181.2+ KB


## Openly_discriminating

In [ ]:
df_rules["Category"].unique()

array(['openly_discriminating', 'unconscious_bias', 'ability-physicality',
       'style', 'gendered', 'inclusive'], dtype=object)

In [ ]:
df_ability = df_rules[df_rules["Category"]=="ability-physicality"]

In [ ]:
df_ability.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 11 entries, 354 to 364
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                11 non-null     object
 1   Word_Type            11 non-null     object
 2   Category             11 non-null     object
 3   Primary_subcategory  11 non-null     object
 4   Sg_all_split         11 non-null     object
 5   Pl_all               11 non-null     object
 6   Pl_all_split         11 non-null     object
 7   Alt_split            11 non-null     object
 8   False_Positives      11 non-null     object
dtypes: object(9)
memory usage: 880.0+ bytes


In [ ]:
df_ability

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
354,fett,a,ability-physicality,advanced_ability,[--- Nicht zum Beschreiben von Personen],--- Nicht zum Beschreiben von Personen,[--- Nicht zum Beschreiben von Personen],[--- Nicht zum Beschreiben von Personen],[]
355,mager,a,ability-physicality,advanced_ability,[--- Nicht zum Beschreiben von Personen],--- Nicht zum Beschreiben von Personen,[--- Nicht zum Beschreiben von Personen],[--- Nicht zum Beschreiben von Personen],[]
356,mollig,a,ability-physicality,advanced_ability,[--- Nicht zum Beschreiben von Personen],--- Nicht zum Beschreiben von Personen,[--- Nicht zum Beschreiben von Personen],[--- Nicht zum Beschreiben von Personen],[]
357,dick,a,ability-physicality,advanced_ability,[--- Nicht zum Beschreiben von Personen],--- Nicht zum Beschreiben von Personen,[--- Nicht zum Beschreiben von Personen],[--- Nicht zum Beschreiben von Personen],[]
358,klein,a,ability-physicality,advanced_ability,[--- Nicht zum Beschreiben von Personen],--- Nicht zum Beschreiben von Personen,[--- Nicht zum Beschreiben von Personen],[--- Nicht zum Beschreiben von Personen],[]
359,hochgewachsen,a,ability-physicality,advanced_ability,[--- Nicht zum Beschreiben von Personen],--- Nicht zum Beschreiben von Personen,[--- Nicht zum Beschreiben von Personen],[--- Nicht zum Beschreiben von Personen],[]
360,hübsch,a,ability-physicality,advanced_ability,[--- Nicht zum Beschreiben von Personen],--- Nicht zum Beschreiben von Personen,[--- Nicht zum Beschreiben von Personen],[--- Nicht zum Beschreiben von Personen],[]
361,gut aussehend,a,ability-physicality,advanced_ability,[--- Nicht zum Beschreiben von Personen],--- Nicht zum Beschreiben von Personen,[--- Nicht zum Beschreiben von Personen],[--- Nicht zum Beschreiben von Personen],[]
362,hässlich,a,ability-physicality,advanced_ability,[--- Nicht zum Beschreiben von Personen],--- Nicht zum Beschreiben von Personen,[--- Nicht zum Beschreiben von Personen],[--- Nicht zum Beschreiben von Personen],[]
363,bildhübsch,a,ability-physicality,advanced_ability,[--- Nicht zum Beschreiben von Personen],--- Nicht zum Beschreiben von Personen,[--- Nicht zum Beschreiben von Personen],[--- Nicht zum Beschreiben von Personen],[]


In [ ]:
df_openly_discrim = df_rules[df_rules["Category"]=="openly_discriminating"]

In [ ]:
#info open_discrim
df_openly_discrim.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 301 entries, 0 to 2635
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                301 non-null    object
 1   Word_Type            301 non-null    object
 2   Category             301 non-null    object
 3   Primary_subcategory  301 non-null    object
 4   Sg_all_split         301 non-null    object
 5   Pl_all               301 non-null    object
 6   Pl_all_split         301 non-null    object
 7   Alt_split            301 non-null    object
 8   False_Positives      301 non-null    object
dtypes: object(9)
memory usage: 23.5+ KB


In [ ]:
df_openly_discrim.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
0,Spast,s,openly_discriminating,ableism,"[Spastiker, -, Person, die mit Spasmen lebt(.....",-,[-],"[-, Spastiker, Person, die mit Spasmen lebt(.....",[]
1,Spasti,s,openly_discriminating,ableism,"[Spastiker, -, Person, die mit Spasmen lebt(.....",-,[-],"[-, Spastiker, Person, die mit Spasmen lebt(.....",[]
2,Albino,s,openly_discriminating,ableism,[-],-,[-],[-],[]
3,Hungerhaken,s,openly_discriminating,ableism,[-],-,[-],[-],[]
4,Krisperl,s,openly_discriminating,ableism,[-],-,[-],[-],[]


In [ ]:
#split dataframe into two words and sentences using function split_words_sentences
df_open_dis_word, df_open_dis_sentence = split_words_sentences_pipe(df_openly_discrim)

In [ ]:
df_open_dis_word.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
0,Spast,s,openly_discriminating,ableism,"[Spastiker, -, Person, die mit Spasmen lebt(.....",-,[-],"[-, Spastiker, Person, die mit Spasmen lebt(.....",[]
1,Spasti,s,openly_discriminating,ableism,"[Spastiker, -, Person, die mit Spasmen lebt(.....",-,[-],"[-, Spastiker, Person, die mit Spasmen lebt(.....",[]
2,Albino,s,openly_discriminating,ableism,[-],-,[-],[-],[]
3,Hungerhaken,s,openly_discriminating,ableism,[-],-,[-],[-],[]
4,Krisperl,s,openly_discriminating,ableism,[-],-,[-],[-],[]


In [ ]:
df_open_dis_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                301 non-null    object
 1   Word_Type            301 non-null    object
 2   Category             301 non-null    object
 3   Primary_subcategory  301 non-null    object
 4   Sg_all_split         301 non-null    object
 5   Pl_all               301 non-null    object
 6   Pl_all_split         301 non-null    object
 7   Alt_split            301 non-null    object
 8   False_Positives      301 non-null    object
dtypes: object(9)
memory usage: 21.3+ KB


In [ ]:
#create subset of dataframe to drop unnecessary columns
df_open_dis_word = df_open_dis_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split", "False_Positives"]]
df_open_dis_sentence = df_open_dis_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split", "False_Positives"]]

In [ ]:
df_open_dis_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 301 entries, 0 to 300
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                301 non-null    object
 1   Word_Type            301 non-null    object
 2   Category             301 non-null    object
 3   Primary_subcategory  301 non-null    object
 4   Alt_split            301 non-null    object
 5   False_Positives      301 non-null    object
dtypes: object(6)
memory usage: 14.2+ KB


In [ ]:
df_open_dis_sentence.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                0 non-null      object
 1   Word_Type            0 non-null      object
 2   Category             0 non-null      object
 3   Primary_subcategory  0 non-null      object
 4   Alt_split            0 non-null      object
 5   False_Positives      0 non-null      object
dtypes: object(6)
memory usage: 0.0+ bytes


In [ ]:
#save to csv file after preprocessing
df_open_dis_word.to_csv("open_dis_words.csv", index=False)

In [ ]:
#save to csv file after preprocessing
df_open_dis_sentence.to_csv("open_dis_sentences.csv", index=False)

## Style

In [ ]:
df_style = df_rules[df_rules["Category"]=="style"]

In [ ]:
#info style
df_style.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 518 entries, 371 to 1787
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                518 non-null    object
 1   Word_Type            518 non-null    object
 2   Category             518 non-null    object
 3   Primary_subcategory  518 non-null    object
 4   Sg_all_split         518 non-null    object
 5   Pl_all               518 non-null    object
 6   Pl_all_split         518 non-null    object
 7   Alt_split            518 non-null    object
 8   False_Positives      518 non-null    object
dtypes: object(9)
memory usage: 40.5+ KB


In [ ]:
#split dataframe into two words and sentences using function split_words_sentences
df_style_word, df_style_sentence = split_words_sentences_pipe(df_style)

In [ ]:
#create subset of dataframe to drop unnecessary columns
df_style_word = df_style_word[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]
df_style_sentence = df_style_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory", "Alt_split"]]

In [ ]:
#save to csv file after preprocessing
df_style_word.to_csv("style_words.csv", index=False)

In [ ]:
#save to csv file after preprocessing
df_style_sentence.to_csv("style_sentences.csv", index=False)

In [ ]:
df_style_sentence.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                0 non-null      object
 1   Word_Type            0 non-null      object
 2   Category             0 non-null      object
 3   Primary_subcategory  0 non-null      object
 4   Alt_split            0 non-null      object
dtypes: object(5)
memory usage: 108.0+ bytes


## Inclusive

In [ ]:
df_inclusive = df_rules[df_rules["Category"]=="inclusive"]

In [ ]:
#info inclusive
df_inclusive.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 313 entries, 1093 to 1423
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                313 non-null    object
 1   Word_Type            313 non-null    object
 2   Category             313 non-null    object
 3   Primary_subcategory  313 non-null    object
 4   Sg_all_split         313 non-null    object
 5   Pl_all               313 non-null    object
 6   Pl_all_split         313 non-null    object
 7   Alt_split            313 non-null    object
 8   False_Positives      313 non-null    object
dtypes: object(9)
memory usage: 24.5+ KB


In [ ]:
df_inclusive.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
1093,lieben,v,inclusive,communal,[],,[],[],[]
1094,abhängig,a,inclusive,communal,[],,[],[],[]
1095,Abhängigkeit,s,inclusive,communal,[],,[],[],[]
1096,Achtsamkeit,s,inclusive,communal,[],,[],[],[]
1097,Anerkennung,s,inclusive,communal,[],,[],[],[]


In [ ]:
df_inclusive["Primary_subcategory"].unique()

array(['communal', 'd_and_i'], dtype=object)

In [ ]:
#split dataframe into two words and sentences using function split_words_sentences
df_inclusive_word, df_inclusive_sentence = split_words_sentences_pipe(df_inclusive)

In [ ]:
#create subset of dataframe to drop unnecessary columns
df_inclusive_word = df_inclusive_word[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]
df_inclusive_sentence = df_inclusive_sentence[['Lemma', "Word_Type", "Category", "Primary_subcategory"]]

In [ ]:
df_inclusive_sentence.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                0 non-null      object
 1   Word_Type            0 non-null      object
 2   Category             0 non-null      object
 3   Primary_subcategory  0 non-null      object
dtypes: object(4)
memory usage: 0.0+ bytes


In [ ]:
df_inclusive_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 313 entries, 0 to 312
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                313 non-null    object
 1   Word_Type            313 non-null    object
 2   Category             313 non-null    object
 3   Primary_subcategory  313 non-null    object
dtypes: object(4)
memory usage: 9.9+ KB


In [ ]:
df_communal_word = df_inclusive_word[df_inclusive_word["Primary_subcategory"]=='communal']

In [ ]:
df_communal_word.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 221 entries, 0 to 220
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                221 non-null    object
 1   Word_Type            221 non-null    object
 2   Category             221 non-null    object
 3   Primary_subcategory  221 non-null    object
dtypes: object(4)
memory usage: 8.6+ KB


In [ ]:
df_communal_sentence = df_inclusive_sentence[df_inclusive_sentence["Primary_subcategory"]=='communal']

In [ ]:
df_communal_sentence.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                0 non-null      object
 1   Word_Type            0 non-null      object
 2   Category             0 non-null      object
 3   Primary_subcategory  0 non-null      object
dtypes: object(4)
memory usage: 0.0+ bytes


In [ ]:
#save to csv file after preprocessing
df_communal_word.to_csv("communal.csv", index=False)

In [ ]:
df_d_and_i_word = df_inclusive_word[df_inclusive_word["Primary_subcategory"]=='d_and_i']

In [ ]:
df_d_and_i_word.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 92 entries, 221 to 312
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                92 non-null     object
 1   Word_Type            92 non-null     object
 2   Category             92 non-null     object
 3   Primary_subcategory  92 non-null     object
dtypes: object(4)
memory usage: 3.6+ KB


In [ ]:
df_d_and_i_sentence = df_inclusive_sentence[df_inclusive_sentence["Primary_subcategory"]=='d_and_i']

In [ ]:
df_d_and_i_sentence.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 4 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                0 non-null      object
 1   Word_Type            0 non-null      object
 2   Category             0 non-null      object
 3   Primary_subcategory  0 non-null      object
dtypes: object(4)
memory usage: 0.0+ bytes


In [ ]:
#save to csv file after preprocessing
df_d_and_i_word.to_csv("d_and_i_words.csv", index=False)

In [ ]:
#save to csv file after preprocessing
df_d_and_i_sentence.to_csv("d_and_i_sentences.csv", index=False)

## Gendered

In [ ]:
df_gendered = df_rules[df_rules["Category"]=="gendered"]

In [ ]:
#info gendered
df_gendered.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 451 entries, 529 to 2587
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                451 non-null    object
 1   Word_Type            451 non-null    object
 2   Category             451 non-null    object
 3   Primary_subcategory  451 non-null    object
 4   Sg_all_split         451 non-null    object
 5   Pl_all               451 non-null    object
 6   Pl_all_split         451 non-null    object
 7   Alt_split            451 non-null    object
 8   False_Positives      451 non-null    object
dtypes: object(9)
memory usage: 35.2+ KB


In [ ]:
#split dataframe into two words and sentences using function split_words_sentences
df_gendered_word, df_gendered_sentence = split_words_sentences_pipe(df_gendered)

In [ ]:
df_gendered_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 451 entries, 0 to 450
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                451 non-null    object
 1   Word_Type            451 non-null    object
 2   Category             451 non-null    object
 3   Primary_subcategory  451 non-null    object
 4   Sg_all_split         451 non-null    object
 5   Pl_all               451 non-null    object
 6   Pl_all_split         451 non-null    object
 7   Alt_split            451 non-null    object
 8   False_Positives      451 non-null    object
dtypes: object(9)
memory usage: 31.8+ KB


In [ ]:
df_gendered_sentence.info()

<class 'pandas.core.frame.DataFrame'>
Index: 0 entries
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                0 non-null      object
 1   Word_Type            0 non-null      object
 2   Category             0 non-null      object
 3   Primary_subcategory  0 non-null      object
 4   Sg_all_split         0 non-null      object
 5   Pl_all               0 non-null      object
 6   Pl_all_split         0 non-null      object
 7   Alt_split            0 non-null      object
 8   False_Positives      0 non-null      object
dtypes: object(9)
memory usage: 0.0+ bytes


In [ ]:
df_gendered_word.query("Word_Type == '-s'")

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives


In [ ]:
df_gendered_word.query("Word_Type == '=s'")

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives


In [ ]:
df_gendered_word_noun = df_gendered_word.query("Word_Type == 's'")

In [ ]:
df_gendered_word_noun.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 433 entries, 0 to 450
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                433 non-null    object
 1   Word_Type            433 non-null    object
 2   Category             433 non-null    object
 3   Primary_subcategory  433 non-null    object
 4   Sg_all_split         433 non-null    object
 5   Pl_all               433 non-null    object
 6   Pl_all_split         433 non-null    object
 7   Alt_split            433 non-null    object
 8   False_Positives      433 non-null    object
dtypes: object(9)
memory usage: 33.8+ KB


In [ ]:
df_gendered_word_noun.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
0,Manpower,s,gendered,advanced_hidden_image,"[Arbeitsleistung, Leistungspotenzial, Personal...",Erwerbspersonen | Arbeitsleistungen | Arbeitsp...,"[Erwerbspersonen, Arbeitsleistungen, Arbeitspo...","[Arbeitsleistung, Leistungspotenzial, Personal...",[]
1,Führerschaft,s,gendered,advanced_hidden_image,"[Leitung, Team-Anleitung, Team-Lenkung, Lenkun...",,[],"[Leitung, Team-Anleitung, Team-Lenkung, Lenkun...",[]
2,Anfängerkurs,s,gendered,advanced_hidden_image,"[Grundkurs, Einstiegskurs]",Grundkurse | Einstiegskurse,"[Grundkurse, Einstiegskurse]","[Grundkurs, Einstiegskurs, Grundkurse, Einstie...",[]
3,anwenderbezogen,s,gendered,advanced_hidden_image,[praxisbezogen],,[],[praxisbezogen],[]
5,Benutzeranleitung,s,gendered,advanced_hidden_image,"[Bedienungsanleitung, Gebrauchsanleitung, Gebr...",Bedienungsanleitungen | Gebrauchsanleitungen |...,"[Bedienungsanleitungen, Gebrauchsanleitungen, ...","[Bedienungsanleitung, Gebrauchsanleitung, Gebr...",[]


In [ ]:
df_gendered_word_noun["Primary_subcategory"].unique()

array(['advanced_hidden_image', 'advanced_titles_base',
       'female_stereotype', 'function_base', 'function', 'leadership',
       'male_stereotype', 'titles', 'titles_base'], dtype=object)

### Sort values in the 'lemma' column by length

In [ ]:
df_gendered_base = df_gendered_word_noun.query("Primary_subcategory == 'advanced_titles_base'")

In [ ]:
# Sort values in the 'lemma' column by length
df_gendered_base = df_gendered_base.iloc[(-df_gendered_base['Lemma'].str.len()).argsort()]

In [ ]:
df_gendered_base.tail()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
103,Maler,s,gendered,advanced_titles_base,[Maler~in~/~Maler],Maler~innen~ und ~Maler,[Maler~innen~ und ~Maler],"[Maler~in~/~Maler, Maler~innen~ und ~Maler]",[]
62,Bauer,s,gendered,advanced_titles_base,"[Bäuer~in~/~Bauer, ~Landbau(...), ~Landwirtsch...",~Landbauende | Bäuer~innen~ und ~Bauern,"[~Landbauende, Bäuer~innen~ und ~Bauern]","[Bäuer~in~/~Bauer, ~Landbau(...), ~Landwirtsch...",[]
57,Agent,s,gendered,advanced_titles_base,"[Agent~in~/~Agent, Agentur für (...), ~Agentur...",Agent~innen~ und ~Agenten,[Agent~innen~ und ~Agenten],"[Agent~in~/~Agent, Agentur für (...), ~Agentur...",[]
96,Koch,s,gendered,advanced_titles_base,"[Köchin~Koch, ~Küche(...), ~Kantine(...)]",Köch~innen~ und ~Köche,[Köch~innen~ und ~Köche],"[Köchin~Koch, ~Küche(...), ~Kantine(...), Köch...",[]
68,Arzt,s,gendered,advanced_titles_base,"[Ärztin~Arzt, medizinisches Team, Praxis(...),...",Ärzt~innen~ und ~Ärzte | medizinisches Team,"[Ärzt~innen~ und ~Ärzte, medizinisches Team]","[Ärztin~Arzt, Praxis(...), Untersuchung(...), ...","[geharzt, geknarzt, gequarzt, verharzt]"


In [ ]:
df_gendered_no_base = df_gendered_word_noun.query("Primary_subcategory != 'advanced_titles_base'")

In [ ]:
# Sort values in the 'lemma' column by length
df_gendered_no_base = df_gendered_no_base.iloc[(-df_gendered_no_base['Lemma'].str.len()).argsort()]

In [ ]:
df_gendered_no_base.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
343,"Anlagenmechaniker Sanitär-, Heizungs- und Kli...",s,gendered,titles,[Anlagenmechaniker~in~/~Anlagenmechaniker Sani...,Anlagenmechaniker~innen~ und ~Anlagenmechanike...,[Anlagenmechaniker~innen~ und ~Anlagenmechanik...,[Anlagenmechaniker~in~/~Anlagenmechaniker Sani...,[]
392,Kaufmann für Spedition und Logistikdienstleistung,s,gendered,titles,[Kauffrau~Kaufmann für Spedition und Logistikd...,Kaufleute für Spedition und Logistikdienstleis...,[Kaufleute für Spedition und Logistikdienstlei...,[Kauffrau~Kaufmann für Spedition und Logistikd...,[]
393,Kaufmann für Versicherungen und Finanzen,s,gendered,titles,[Kauffrau~Kaufmann für Versicherungen und Fina...,Kaufleute für Versicherungen und Finanzen,[Kaufleute für Versicherungen und Finanzen],[Kauffrau~Kaufmann für Versicherungen und Fina...,[]
411,Medizinisch-technischer Laborassistent,s,gendered,titles,[Medizinisch-technische~r Laborassistent~in~/~...,Medizinisch-technische Laborassistent~innen~ u...,[Medizinisch-technische Laborassistent~innen~ ...,[Medizinisch-technische~r Laborassistent~in~/~...,[]
368,Fachangestellter für Bürokommunikation,s,gendered,titles,[Fachangestellte~Fachangestellter für Bürokomm...,Fachangestellte für Bürokommunikation,[Fachangestellte für Bürokommunikation],[Fachangestellte~Fachangestellter für Bürokomm...,[]


In [ ]:
sorted_df_gendered_word_noun = pd.concat([df_gendered_no_base, df_gendered_base])

In [ ]:
# Reset the index of the sorted DataFrame
sorted_df_gendered_word_noun = sorted_df_gendered_word_noun.reset_index(drop=True)

In [ ]:
sorted_df_gendered_word_noun.head()

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
0,"Anlagenmechaniker Sanitär-, Heizungs- und Kli...",s,gendered,titles,[Anlagenmechaniker~in~/~Anlagenmechaniker Sani...,Anlagenmechaniker~innen~ und ~Anlagenmechanike...,[Anlagenmechaniker~innen~ und ~Anlagenmechanik...,[Anlagenmechaniker~in~/~Anlagenmechaniker Sani...,[]
1,Kaufmann für Spedition und Logistikdienstleistung,s,gendered,titles,[Kauffrau~Kaufmann für Spedition und Logistikd...,Kaufleute für Spedition und Logistikdienstleis...,[Kaufleute für Spedition und Logistikdienstlei...,[Kauffrau~Kaufmann für Spedition und Logistikd...,[]
2,Kaufmann für Versicherungen und Finanzen,s,gendered,titles,[Kauffrau~Kaufmann für Versicherungen und Fina...,Kaufleute für Versicherungen und Finanzen,[Kaufleute für Versicherungen und Finanzen],[Kauffrau~Kaufmann für Versicherungen und Fina...,[]
3,Medizinisch-technischer Laborassistent,s,gendered,titles,[Medizinisch-technische~r Laborassistent~in~/~...,Medizinisch-technische Laborassistent~innen~ u...,[Medizinisch-technische Laborassistent~innen~ ...,[Medizinisch-technische~r Laborassistent~in~/~...,[]
4,Fachangestellter für Bürokommunikation,s,gendered,titles,[Fachangestellte~Fachangestellter für Bürokomm...,Fachangestellte für Bürokommunikation,[Fachangestellte für Bürokommunikation],[Fachangestellte~Fachangestellter für Bürokomm...,[]


In [ ]:
df_gendered_word_no_noun = df_gendered_word.query("Word_Type != 's'")

In [ ]:
#drop unnecessary column "Pl_all"
sorted_df_gendered_word_noun = sorted_df_gendered_word_noun.drop(labels=["Pl_all", "False_Positives"], axis=1)
df_gendered_word_no_noun = df_gendered_word_no_noun.drop(labels=["Pl_all", "False_Positives"], axis=1)
df_gendered_sentence = df_gendered_sentence.drop(labels=["Pl_all", "False_Positives"], axis=1)

In [ ]:
#save to csv file after preprocessing
sorted_df_gendered_word_noun.to_csv("gendered_noun_words.csv", index=False)

In [ ]:
#save to csv file after preprocessing
df_gendered_word_no_noun.to_csv("gendered_no_noun_words.csv", index=False)

In [ ]:
df_gendered_sentence.to_csv("gendered_sentences.csv", index=False)

## Unconscious Bias

In [ ]:
df_ub = df_rules[df_rules["Category"]=='unconscious_bias']

In [ ]:
df_ub = df_rules[(df_rules["Category"]=='unconscious_bias') | (df_rules["Category"]=='ability-physicality')]

In [ ]:
#info unc_bias
df_ub.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 736 entries, 300 to 2614
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                736 non-null    object
 1   Word_Type            736 non-null    object
 2   Category             736 non-null    object
 3   Primary_subcategory  736 non-null    object
 4   Sg_all_split         736 non-null    object
 5   Pl_all               736 non-null    object
 6   Pl_all_split         736 non-null    object
 7   Alt_split            736 non-null    object
 8   False_Positives      736 non-null    object
dtypes: object(9)
memory usage: 57.5+ KB


In [ ]:
#split dataframe into two words and sentences using function split_words_sentences
df_ub_word, df_ub_sentence = split_words_sentences_pipe(df_ub)

In [ ]:
df_ub_word.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736 entries, 0 to 735
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                736 non-null    object
 1   Word_Type            736 non-null    object
 2   Category             736 non-null    object
 3   Primary_subcategory  736 non-null    object
 4   Sg_all_split         736 non-null    object
 5   Pl_all               736 non-null    object
 6   Pl_all_split         736 non-null    object
 7   Alt_split            736 non-null    object
 8   False_Positives      736 non-null    object
dtypes: object(9)
memory usage: 51.9+ KB


In [ ]:
df_ub_word[df_ub_word["Word_Type"]=='-s']

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives


In [ ]:
df_ub_word[df_ub_word["Word_Type"]=='=s']

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
378,Verrückter,=s,unconscious_bias,cognitive_perception,"[Mensch mit psychischer Störung(...), jemand, ...",,[],"[Mensch mit psychischer Störung(...), jemand, ...",[]
379,Wahnsinnige,=s,unconscious_bias,cognitive_perception,"[Mensch mit psychischer Störung(...), jemand, ...",,[],"[Mensch mit psychischer Störung(...), jemand, ...",[]
380,Wahnsinniger,=s,unconscious_bias,cognitive_perception,"[Mensch mit psychischer Störung(...), jemand, ...",,[],"[Mensch mit psychischer Störung(...), jemand, ...",[]
383,Geistesgestörter,=s,unconscious_bias,cognitive_perception,[Mensch mit psychischer Störung(...)],,[],[Mensch mit psychischer Störung(...)],[]
393,Dunkelhäutige,=s,unconscious_bias,color,"[Person, Mensch, Person of Color, Black Person...",Personen | Menschen | People of Color | BIPoC | -,"[Personen, Menschen, People of Color, BIPoC, -]","[-, Person, Mensch, Person of Color, Black Per...",[]
394,Dunkelhäutiger,=s,unconscious_bias,color,"[Person, Mensch, Person of Color, Black Person...",Personen | Menschen | People of Color | BIPoC | -,"[Personen, Menschen, People of Color, BIPoC, -]","[-, Person, Mensch, Person of Color, Black Per...",[]
396,Farbige,=s,unconscious_bias,color,"[Person, Mensch, Person of Color, Black Person...",Personen | Menschen | People of Color | BIPoC ...,"[Personen, Menschen, People of Color, BIPoC, -...","[--- nicht auf Menschen beziehen, Person, Mens...",[]
397,Farbiger,=s,unconscious_bias,color,"[Person, Mensch, Person of Color, Black Person...",Personen | Menschen | People of Color | BIPoC ...,"[Personen, Menschen, People of Color, BIPoC, -...","[--- nicht auf Menschen beziehen, Person, Mens...",[]
400,Hellhäutige,=s,unconscious_bias,color,"[Person, Mensch, -]",-,[-],"[-, Person, Mensch]",[]
401,Hellhäutiger,=s,unconscious_bias,color,"[Person, Mensch, -]",-,[-],"[-, Person, Mensch]",[]


In [ ]:
df_ub_word[df_ub_word["Word_Type"]=='~s']

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives


In [ ]:
#filter datframe to have plural for subj(noun) without special caracters and empty strings
df_ub_word_pl_1 = df_ub_word[(df_ub_word["Pl_all"]!='')&(df_ub_word["Pl_all"]!='-')&(df_ub_word["Word_Type"]=='s')]

In [ ]:
#filter datframe to have plural for subj(noun) without special caracters and empty strings
df_ub_word_pl_2 = df_ub_word[(df_ub_word["Pl_all"]!='')&(df_ub_word["Pl_all"]!='-')&(df_ub_word["Word_Type"]=='=s')]

In [ ]:
df_ub_word_pl = pd.concat([df_ub_word_pl_1, df_ub_word_pl_2])

In [ ]:
df_ub_word_pl_1.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 141 entries, 0 to 720
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                141 non-null    object
 1   Word_Type            141 non-null    object
 2   Category             141 non-null    object
 3   Primary_subcategory  141 non-null    object
 4   Sg_all_split         141 non-null    object
 5   Pl_all               141 non-null    object
 6   Pl_all_split         141 non-null    object
 7   Alt_split            141 non-null    object
 8   False_Positives      141 non-null    object
dtypes: object(9)
memory usage: 11.0+ KB


In [ ]:
df_ub_word_pl_1

,Lemma,Word_Type,Category,Primary_subcategory,Sg_all_split,Pl_all,Pl_all_split,Alt_split,False_Positives
393,Dunkelhäutige,=s,unconscious_bias,color,"[Person, Mensch, Person of Color, Black Person...",Personen | Menschen | People of Color | BIPoC | -,"[Personen, Menschen, People of Color, BIPoC, -]","[-, Person, Mensch, Person of Color, Black Per...",[]
394,Dunkelhäutiger,=s,unconscious_bias,color,"[Person, Mensch, Person of Color, Black Person...",Personen | Menschen | People of Color | BIPoC | -,"[Personen, Menschen, People of Color, BIPoC, -]","[-, Person, Mensch, Person of Color, Black Per...",[]
396,Farbige,=s,unconscious_bias,color,"[Person, Mensch, Person of Color, Black Person...",Personen | Menschen | People of Color | BIPoC ...,"[Personen, Menschen, People of Color, BIPoC, -...","[--- nicht auf Menschen beziehen, Person, Mens...",[]
397,Farbiger,=s,unconscious_bias,color,"[Person, Mensch, Person of Color, Black Person...",Personen | Menschen | People of Color | BIPoC ...,"[Personen, Menschen, People of Color, BIPoC, -...","[--- nicht auf Menschen beziehen, Person, Mens...",[]


In [ ]:
df_ub_word_pl.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 145 entries, 0 to 397
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                145 non-null    object
 1   Word_Type            145 non-null    object
 2   Category             145 non-null    object
 3   Primary_subcategory  145 non-null    object
 4   Sg_all_split         145 non-null    object
 5   Pl_all               145 non-null    object
 6   Pl_all_split         145 non-null    object
 7   Alt_split            145 non-null    object
 8   False_Positives      145 non-null    object
dtypes: object(9)
memory usage: 11.3+ KB


#### save the left data into separate dataframe

In [ ]:
plural_list = list(df_ub_word_pl["Lemma"])

In [ ]:
df_ub_word_left = df_ub_word.query("Lemma != @plural_list")

#### remove unnecessary column "Pl_all"

In [ ]:
df_ub_word_left = df_ub_word_left.drop(labels=["Pl_all", "False_Positives"], axis=1)
df_ub_word_pl = df_ub_word_pl.drop(labels=["Pl_all", "False_Positives"], axis=1)
df_ub_sentence = df_ub_sentence.drop(labels=["Pl_all", "False_Positives"], axis=1)

#### save to csv files after preprocessing

In [ ]:
#save to csv files after preprocessing
df_ub_word_left.to_csv("ub_no_plur_words.csv", index=False)
df_ub_word_pl.to_csv("ub_plur_words.csv", index=False)
df_ub_sentence.to_csv("ub_sentences.csv", index=False)

In [ ]:
df_test = pd.read_csv("ub_no_plur_words.csv")

In [ ]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 591 entries, 0 to 590
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Lemma                591 non-null    object
 1   Word_Type            591 non-null    object
 2   Category             591 non-null    object
 3   Primary_subcategory  591 non-null    object
 4   Sg_all_split         591 non-null    object
 5   Pl_all_split         591 non-null    object
 6   Alt_split            591 non-null    object
dtypes: object(7)
memory usage: 32.4+ KB
